# Multi-GPU Mirrored Stategy in Tensorflow

- Mirrored Strategy in Tensorflow is a data parallelism strategy that allows training on multiple GPUs by mirroring(replicating) the model across them. It synchronizes the weights across all GPUs during traing, ensuring that each GPU has the same model parameters at each step.

## How it works

1. Model replication
- The model is copied onto each GPU. If you have, for example, two GPUs, both will have the same initial model parametes.
2. Data Parallelism
- The input is divided into batches and distributed equally among all GPUs. If the batch size is 64 and you have 2 GPUs, each GPU gets a batch of 32. 
3. Synchronous Training
- Each GPU computes forward and backward passes on its own batch.
- Gradients from all GPUs are averaged and synchronized.
- The model parameters are updated synchronously across all GPUs to maintain consistency. 
4. All Reduce operations
- Tensorflow uses an all reduce algorithm to aggregate gradients across GPUs efficiently.
- This ensures that each GPU applies the same parameter updates. 

In [ ]:
import tensorflow as tf

# Create MirroredStrategy
strategy = tf.distribute.MirroredStrategy()

# Define and compile model inside strategy scope
with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation='relu', input_shape=(784,)),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

# Load data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

# Train the model
model.fit(x_train, y_train, batch_size=64, epochs=5)



# When to use mirrored strategy ?
- If you have multiple GPUs on a single machine
- If you need synchronous training with data parallelism.
- When training large models that would benefit from distributing computation. 

# Alternative to mirrored strategy
- tf.distribute.MultiWorkerMirroredStrategy: For multiple machines with GPUs.
- tf.distribute.TPUStrategy: For training on TPUs.
- tf.distribute.ParameterServerStrategy: For distributed asynchronous training. 